In [12]:
import pandas as pd

years = [2022, 2023, 2024, 2025, 2026]

dfs = []

for year in years:
    df = pd.read_csv(f"../data/processed/cleaned_{year}.csv")
    df["Arrival_Date"] = pd.to_datetime(df["Arrival_Date"])
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

daily_counts = (
    df.groupby(
        ["Commodity", "Market", "Arrival_Date"]
    )
    .size()
    .reset_index(name="Observation_Count")
)

print(daily_counts["Observation_Count"].value_counts().sort_index())
print("\nMaximum observations in one market-commodity-day:",
      daily_counts["Observation_Count"].max())

Observation_Count
1    2349876
2      79602
3       2698
4       1351
5         65
6          7
7          1
Name: count, dtype: int64

Maximum observations in one market-commodity-day: 7


In [13]:
daily_df = (
    df.groupby(
        ["Commodity", "Market", "Arrival_Date"],
        as_index=False
    )
    .agg(
        Daily_Modal_Price=("Modal_Price", "median")
    )
)

print("Daily dataset shape:", daily_df.shape)

print("\nColumns:")
print(daily_df.columns.tolist())

print("\nSample:")
print(daily_df.head())

print("\nMissing values:")
print(daily_df.isna().sum())

Daily dataset shape: (2433600, 4)

Columns:
['Commodity', 'Market', 'Arrival_Date', 'Daily_Modal_Price']

Sample:
  Commodity Market Arrival_Date  Daily_Modal_Price
0     Onion  A lot   2022-05-24              280.0
1     Onion  A lot   2022-05-25              280.0
2     Onion  A lot   2022-05-26              200.0
3     Onion  A lot   2022-05-27              245.0
4     Onion  A lot   2022-05-28              245.0

Missing values:
Commodity            0
Market               0
Arrival_Date         0
Daily_Modal_Price    0
dtype: int64


In [14]:
daily_df = daily_df.sort_values(
    ["Commodity", "Market", "Arrival_Date"]
).copy()

daily_df["Price_Return"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .pct_change()
)

print(daily_df[[
    "Commodity",
    "Market",
    "Arrival_Date",
    "Daily_Modal_Price",
    "Price_Return"
]].head(15))

   Commodity Market Arrival_Date  Daily_Modal_Price  Price_Return
0      Onion  A lot   2022-05-24              280.0           NaN
1      Onion  A lot   2022-05-25              280.0      0.000000
2      Onion  A lot   2022-05-26              200.0     -0.285714
3      Onion  A lot   2022-05-27              245.0      0.225000
4      Onion  A lot   2022-05-28              245.0      0.000000
5      Onion  A lot   2022-05-29              220.0     -0.102041
6      Onion  A lot   2022-05-30              270.0      0.227273
7      Onion  A lot   2022-05-31              200.0     -0.259259
8      Onion  A lot   2022-06-01              245.0      0.225000
9      Onion  A lot   2022-06-02              401.0      0.636735
10     Onion  A lot   2022-06-03              150.0     -0.625935
11     Onion  A lot   2022-06-04              300.0      1.000000
12     Onion  A lot   2022-06-05              300.0      0.000000
13     Onion  A lot   2022-06-06              177.0     -0.410000
14     Oni

In [15]:
daily_df["Rolling_Volatility_7D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Price_Return"]
    .transform(
        lambda x: x.rolling(window=7, min_periods=7).std()
    )
)

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Price_Return",
            "Rolling_Volatility_7D"
        ]
    ].head(20)
)

print("\nMissing rolling volatility:")
print(daily_df["Rolling_Volatility_7D"].isna().sum())

   Commodity Market Arrival_Date  Price_Return  Rolling_Volatility_7D
0      Onion  A lot   2022-05-24           NaN                    NaN
1      Onion  A lot   2022-05-25      0.000000                    NaN
2      Onion  A lot   2022-05-26     -0.285714                    NaN
3      Onion  A lot   2022-05-27      0.225000                    NaN
4      Onion  A lot   2022-05-28      0.000000                    NaN
5      Onion  A lot   2022-05-29     -0.102041                    NaN
6      Onion  A lot   2022-05-30      0.227273                    NaN
7      Onion  A lot   2022-05-31     -0.259259               0.206608
8      Onion  A lot   2022-06-01      0.225000               0.228047
9      Onion  A lot   2022-06-02      0.636735               0.290487
10     Onion  A lot   2022-06-03     -0.625935               0.403248
11     Onion  A lot   2022-06-04      1.000000               0.548288
12     Onion  A lot   2022-06-05      0.000000               0.541556
13     Onion  A lot 

In [16]:
volatility = daily_df["Rolling_Volatility_7D"].dropna()

print(volatility.describe())

print("\nSelected percentiles:")
print(
    volatility.quantile(
        [0.50, 0.75, 0.80, 0.85, 0.90, 0.95, 0.99]
    )
)

count    2.384538e+06
mean     7.033080e-01
std      3.212583e+02
min      0.000000e+00
25%      2.519763e-02
50%      6.711238e-02
75%      1.359173e-01
max      1.874678e+05
Name: Rolling_Volatility_7D, dtype: float64

Selected percentiles:
0.50    0.067112
0.75    0.135917
0.80    0.159239
0.85    0.189439
0.90    0.237256
0.95    0.330919
0.99    0.694478
Name: Rolling_Volatility_7D, dtype: float64


In [17]:
p99 = volatility.quantile(0.99)

extreme_vol = daily_df[
    daily_df["Rolling_Volatility_7D"] > p99
]

print("99th percentile:", p99)

print("\nExtreme observations:", len(extreme_vol))

print("\nBy commodity:")
print(
    extreme_vol["Commodity"]
    .value_counts()
)

print("\nTop 10 extreme volatility values:")
print(
    extreme_vol[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Rolling_Volatility_7D"
        ]
    ]
    .sort_values(
        "Rolling_Volatility_7D",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)

99th percentile: 0.694478354869655

Extreme observations: 23846

By commodity:
Commodity
Tomato    11045
Onion      8665
Potato     4136
Name: count, dtype: int64

Top 10 extreme volatility values:
Commodity      Market Arrival_Date  Rolling_Volatility_7D
    Onion      Kalvan   2024-12-24          187467.750274
    Onion      Kalvan   2024-12-27          187467.749178
    Onion      Kalvan   2024-12-28          187467.710439
    Onion      Kalvan   2024-12-23          187467.694140
    Onion      Kalvan   2024-12-30          187467.690306
    Onion      Kalvan   2025-01-02          187467.681546
    Onion      Kalvan   2025-01-03          187467.666268
   Potato Mapusa APMC   2026-01-23            3897.396617
   Potato Mapusa APMC   2026-01-26            3897.385017
    Onion Mapusa APMC   2026-01-15            3455.346233


In [18]:
returns = daily_df["Price_Return"].dropna()

print(returns.describe())

print("\nSelected return percentiles:")
print(
    returns.quantile(
        [0.01, 0.05, 0.50, 0.95, 0.99]
    )
)

count    2.426135e+06
mean     2.465336e-01
std      3.186693e+02
min     -9.999983e-01
25%     -1.588068e-02
50%      0.000000e+00
75%      1.398601e-02
max      4.959928e+05
Name: Price_Return, dtype: float64

Selected return percentiles:
0.01   -0.365854
0.05   -0.181818
0.50    0.000000
0.95    0.217105
0.99    0.590909
Name: Price_Return, dtype: float64


In [19]:
extreme_returns = (
    daily_df[
        daily_df["Price_Return"].abs() > 5
    ]
    [
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Price_Return"
        ]
    ]
    .sort_values(
        "Price_Return",
        key=lambda x: x.abs(),
        ascending=False
    )
)

print("Number of |return| > 5 observations:", len(extreme_returns))

print("\nTop 20:")
print(extreme_returns.head(20).to_string(index=False))

Number of |return| > 5 observations: 732

Top 20:
Commodity                           Market Arrival_Date  Daily_Modal_Price  Price_Return
    Onion                           Kalvan   2024-12-23        917588483.0 495992.774595
   Potato                      Mapusa APMC   2025-12-24             1650.0  10311.500000
   Potato                   Kalimpong APMC   2026-02-08             2600.0   9999.000000
    Onion                      Mapusa APMC   2025-12-23             1600.0   9141.857143
   Tomato Pavilion Ground,Khammam,RBZ APMC   2026-01-04             3800.0   7306.692308
   Tomato  Sattupalli (ramalayam),RBZ APMC   2026-04-05             2000.0   4165.666667
    Onion                      Mapusa APMC   2025-12-10              130.0   1129.434783
    Onion             Junnar (Narayangaon)   2022-10-15             1500.0    249.000000
   Tomato                       Patti APMC   2026-02-03             2850.0    225.190476
    Onion                       A lot APMC   2025-12-05     

In [20]:
return_abs = daily_df["Price_Return"].abs()

print("Absolute return thresholds:")

for threshold in [1, 2, 5, 10, 50, 100]:
    print(
        f"|return| > {threshold}:",
        (return_abs > threshold).sum()
    )

Absolute return thresholds:
|return| > 1: 7888
|return| > 2: 2357
|return| > 5: 732
|return| > 10: 389
|return| > 50: 323
|return| > 100: 88


In [21]:
lower_return = daily_df["Price_Return"].quantile(0.01)
upper_return = daily_df["Price_Return"].quantile(0.99)

daily_df["Return_Capped"] = (
    daily_df["Price_Return"]
    .clip(
        lower=lower_return,
        upper=upper_return
    )
)

print("Lower cap:", lower_return)
print("Upper cap:", upper_return)

print("\nOriginal return:")
print(daily_df["Price_Return"].describe())

print("\nCapped return:")
print(daily_df["Return_Capped"].describe())

Lower cap: -0.36585365853658536
Upper cap: 0.5909090909090908

Original return:
count    2.426135e+06
mean     2.465336e-01
std      3.186693e+02
min     -9.999983e-01
25%     -1.588068e-02
50%      0.000000e+00
75%      1.398601e-02
max      4.959928e+05
Name: Price_Return, dtype: float64

Capped return:
count    2.426135e+06
mean     6.798779e-03
std      1.248567e-01
min     -3.658537e-01
25%     -1.588068e-02
50%      0.000000e+00
75%      1.398601e-02
max      5.909091e-01
Name: Return_Capped, dtype: float64


In [22]:
daily_df["Rolling_Volatility_7D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Return_Capped"]
    .transform(
        lambda x: x.rolling(window=7, min_periods=7).std()
    )
)

robust_volatility = daily_df["Rolling_Volatility_7D"].dropna()

print(robust_volatility.describe())

print("\nSelected percentiles:")
print(
    robust_volatility.quantile(
        [0.50, 0.75, 0.80, 0.85, 0.90, 0.95, 0.99]
    )
)

count    2.384538e+06
mean     9.364687e-02
std      8.885700e-02
min      0.000000e+00
25%      2.519763e-02
50%      6.711238e-02
75%      1.357453e-01
max      5.114112e-01
Name: Rolling_Volatility_7D, dtype: float64

Selected percentiles:
0.50    0.067112
0.75    0.135745
0.80    0.157359
0.85    0.186764
0.90    0.227855
0.95    0.283770
0.99    0.367572
Name: Rolling_Volatility_7D, dtype: float64


In [23]:
p75 = robust_volatility.quantile(0.75)
p90 = robust_volatility.quantile(0.90)

daily_df["Risk_Level"] = pd.cut(
    daily_df["Rolling_Volatility_7D"],
    bins=[-float("inf"), p75, p90, float("inf")],
    labels=["Low", "Medium", "High"]
)

print("Thresholds:")
print("Low / Medium:", p75)
print("Medium / High:", p90)

print("\nTarget distribution:")
print(daily_df["Risk_Level"].value_counts(dropna=False))

print("\nTarget percentage:")
print(
    daily_df["Risk_Level"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)


Thresholds:
Low / Medium: 0.13574531918384466
Medium / High: 0.22785492405866373

Target distribution:
Risk_Level
Low       1788405
Medium     357679
High       238454
NaN         49062
Name: count, dtype: int64

Target percentage:
Risk_Level
Low       73.49
Medium    14.70
High       9.80
NaN        2.02
Name: proportion, dtype: float64


In [24]:
daily_df["Future_Volatility_7D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Return_Capped"]
    .transform(
        lambda x: x.iloc[::-1]
        .rolling(window=7, min_periods=7)
        .std()
        .iloc[::-1]
    )
)

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Rolling_Volatility_7D",
            "Future_Volatility_7D"
        ]
    ].head(20)
)

print("\nMissing future volatility:")
print(daily_df["Future_Volatility_7D"].isna().sum())

   Commodity Market Arrival_Date  Rolling_Volatility_7D  Future_Volatility_7D
0      Onion  A lot   2022-05-24                    NaN                   NaN
1      Onion  A lot   2022-05-25                    NaN              0.206608
2      Onion  A lot   2022-05-26                    NaN              0.228047
3      Onion  A lot   2022-05-27                    NaN              0.277553
4      Onion  A lot   2022-05-28                    NaN              0.328801
5      Onion  A lot   2022-05-29                    NaN              0.386134
6      Onion  A lot   2022-05-30                    NaN              0.377769
7      Onion  A lot   2022-05-31               0.206608              0.420155
8      Onion  A lot   2022-06-01               0.228047              0.435291
9      Onion  A lot   2022-06-02               0.277553              0.480268
10     Onion  A lot   2022-06-03               0.328801              0.427956
11     Onion  A lot   2022-06-04               0.386134         

In [25]:
p75 = robust_volatility.quantile(0.75)
p90 = robust_volatility.quantile(0.90)

daily_df["Future_Risk_Level"] = pd.cut(
    daily_df["Future_Volatility_7D"],
    bins=[-float("inf"), p75, p90, float("inf")],
    labels=["Low", "Medium", "High"]
)

print("Final target thresholds:")
print("Low / Medium:", p75)
print("Medium / High:", p90)

print("\nFuture target distribution:")
print(
    daily_df["Future_Risk_Level"]
    .value_counts(dropna=False)
)

print("\nFuture target percentage:")
print(
    daily_df["Future_Risk_Level"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

Final target thresholds:
Low / Medium: 0.13574531918384466
Medium / High: 0.22785492405866373

Future target distribution:
Future_Risk_Level
Low       1788405
Medium     357679
High       238454
NaN         49062
Name: count, dtype: int64

Future target percentage:
Future_Risk_Level
Low       73.49
Medium    14.70
High       9.80
NaN        2.02
Name: proportion, dtype: float64


In [26]:
daily_df["Lag_1_Price"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .shift(1)
)

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Lag_1_Price",
            "Future_Risk_Level"
        ]
    ].head(15)
)

print("\nMissing Lag_1_Price:")
print(daily_df["Lag_1_Price"].isna().sum())

   Commodity Market Arrival_Date  Daily_Modal_Price  Lag_1_Price  \
0      Onion  A lot   2022-05-24              280.0          NaN   
1      Onion  A lot   2022-05-25              280.0        280.0   
2      Onion  A lot   2022-05-26              200.0        280.0   
3      Onion  A lot   2022-05-27              245.0        200.0   
4      Onion  A lot   2022-05-28              245.0        245.0   
5      Onion  A lot   2022-05-29              220.0        245.0   
6      Onion  A lot   2022-05-30              270.0        220.0   
7      Onion  A lot   2022-05-31              200.0        270.0   
8      Onion  A lot   2022-06-01              245.0        200.0   
9      Onion  A lot   2022-06-02              401.0        245.0   
10     Onion  A lot   2022-06-03              150.0        401.0   
11     Onion  A lot   2022-06-04              300.0        150.0   
12     Onion  A lot   2022-06-05              300.0        300.0   
13     Onion  A lot   2022-06-06              17

In [27]:
daily_df["Lag_3_Price"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .shift(3)
)

daily_df["Lag_7_Price"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .shift(7)
)

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Lag_1_Price",
            "Lag_3_Price",
            "Lag_7_Price"
        ]
    ].head(15)
)

print("\nMissing values:")
print(
    daily_df[
        ["Lag_1_Price", "Lag_3_Price", "Lag_7_Price"]
    ].isna().sum()
)

   Commodity Market Arrival_Date  Daily_Modal_Price  Lag_1_Price  Lag_3_Price  \
0      Onion  A lot   2022-05-24              280.0          NaN          NaN   
1      Onion  A lot   2022-05-25              280.0        280.0          NaN   
2      Onion  A lot   2022-05-26              200.0        280.0          NaN   
3      Onion  A lot   2022-05-27              245.0        200.0        280.0   
4      Onion  A lot   2022-05-28              245.0        245.0        280.0   
5      Onion  A lot   2022-05-29              220.0        245.0        200.0   
6      Onion  A lot   2022-05-30              270.0        220.0        245.0   
7      Onion  A lot   2022-05-31              200.0        270.0        245.0   
8      Onion  A lot   2022-06-01              245.0        200.0        220.0   
9      Onion  A lot   2022-06-02              401.0        245.0        270.0   
10     Onion  A lot   2022-06-03              150.0        401.0        200.0   
11     Onion  A lot   2022-0

In [28]:
daily_df["Price_Change_1D"] = (
    daily_df["Daily_Modal_Price"] / daily_df["Lag_1_Price"]
) - 1

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Lag_1_Price",
            "Price_Change_1D"
        ]
    ].head(15)
)

print("\nMissing Price_Change_1D:")
print(daily_df["Price_Change_1D"].isna().sum())

print("\nPrice_Change_1D summary:")
print(daily_df["Price_Change_1D"].describe())

   Commodity Market Arrival_Date  Daily_Modal_Price  Lag_1_Price  \
0      Onion  A lot   2022-05-24              280.0          NaN   
1      Onion  A lot   2022-05-25              280.0        280.0   
2      Onion  A lot   2022-05-26              200.0        280.0   
3      Onion  A lot   2022-05-27              245.0        200.0   
4      Onion  A lot   2022-05-28              245.0        245.0   
5      Onion  A lot   2022-05-29              220.0        245.0   
6      Onion  A lot   2022-05-30              270.0        220.0   
7      Onion  A lot   2022-05-31              200.0        270.0   
8      Onion  A lot   2022-06-01              245.0        200.0   
9      Onion  A lot   2022-06-02              401.0        245.0   
10     Onion  A lot   2022-06-03              150.0        401.0   
11     Onion  A lot   2022-06-04              300.0        150.0   
12     Onion  A lot   2022-06-05              300.0        300.0   
13     Onion  A lot   2022-06-06              17

In [29]:
daily_df["Price_Change_3D"] = (
    daily_df["Daily_Modal_Price"] / daily_df["Lag_3_Price"]
) - 1

daily_df["Price_Change_7D"] = (
    daily_df["Daily_Modal_Price"] / daily_df["Lag_7_Price"]
) - 1

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Lag_1_Price",
            "Lag_3_Price",
            "Lag_7_Price",
            "Price_Change_1D",
            "Price_Change_3D",
            "Price_Change_7D"
        ]
    ].head(15)
)

print("\nMissing values:")
print(
    daily_df[
        [
            "Price_Change_1D",
            "Price_Change_3D",
            "Price_Change_7D"
        ]
    ].isna().sum()
)

   Commodity Market Arrival_Date  Daily_Modal_Price  Lag_1_Price  Lag_3_Price  \
0      Onion  A lot   2022-05-24              280.0          NaN          NaN   
1      Onion  A lot   2022-05-25              280.0        280.0          NaN   
2      Onion  A lot   2022-05-26              200.0        280.0          NaN   
3      Onion  A lot   2022-05-27              245.0        200.0        280.0   
4      Onion  A lot   2022-05-28              245.0        245.0        280.0   
5      Onion  A lot   2022-05-29              220.0        245.0        200.0   
6      Onion  A lot   2022-05-30              270.0        220.0        245.0   
7      Onion  A lot   2022-05-31              200.0        270.0        245.0   
8      Onion  A lot   2022-06-01              245.0        200.0        220.0   
9      Onion  A lot   2022-06-02              401.0        245.0        270.0   
10     Onion  A lot   2022-06-03              150.0        401.0        200.0   
11     Onion  A lot   2022-0

In [30]:
daily_df["Rolling_Mean_7D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .transform(
        lambda x: x.shift(1).rolling(window=7, min_periods=7).mean()
    )
)

daily_df["Rolling_Std_7D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .transform(
        lambda x: x.shift(1).rolling(window=7, min_periods=7).std()
    )
)

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Rolling_Mean_7D",
            "Rolling_Std_7D"
        ]
    ].head(15)
)

print("\nMissing values:")
print(
    daily_df[
        ["Rolling_Mean_7D", "Rolling_Std_7D"]
    ].isna().sum()
)

   Commodity Market Arrival_Date  Daily_Modal_Price  Rolling_Mean_7D  \
0      Onion  A lot   2022-05-24              280.0              NaN   
1      Onion  A lot   2022-05-25              280.0              NaN   
2      Onion  A lot   2022-05-26              200.0              NaN   
3      Onion  A lot   2022-05-27              245.0              NaN   
4      Onion  A lot   2022-05-28              245.0              NaN   
5      Onion  A lot   2022-05-29              220.0              NaN   
6      Onion  A lot   2022-05-30              270.0              NaN   
7      Onion  A lot   2022-05-31              200.0       248.571429   
8      Onion  A lot   2022-06-01              245.0       237.142857   
9      Onion  A lot   2022-06-02              401.0       232.142857   
10     Onion  A lot   2022-06-03              150.0       260.857143   
11     Onion  A lot   2022-06-04              300.0       247.285714   
12     Onion  A lot   2022-06-05              300.0       255.14

In [31]:
daily_df["Rolling_Mean_14D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .transform(
        lambda x: x.shift(1).rolling(window=14, min_periods=14).mean()
    )
)

daily_df["Rolling_Std_14D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .transform(
        lambda x: x.shift(1).rolling(window=14, min_periods=14).std()
    )
)

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Rolling_Mean_14D",
            "Rolling_Std_14D"
        ]
    ].head(20)
)

print("\nMissing values:")
print(
    daily_df[
        ["Rolling_Mean_14D", "Rolling_Std_14D"]
    ].isna().sum()
)

   Commodity Market Arrival_Date  Daily_Modal_Price  Rolling_Mean_14D  \
0      Onion  A lot   2022-05-24              280.0               NaN   
1      Onion  A lot   2022-05-25              280.0               NaN   
2      Onion  A lot   2022-05-26              200.0               NaN   
3      Onion  A lot   2022-05-27              245.0               NaN   
4      Onion  A lot   2022-05-28              245.0               NaN   
5      Onion  A lot   2022-05-29              220.0               NaN   
6      Onion  A lot   2022-05-30              270.0               NaN   
7      Onion  A lot   2022-05-31              200.0               NaN   
8      Onion  A lot   2022-06-01              245.0               NaN   
9      Onion  A lot   2022-06-02              401.0               NaN   
10     Onion  A lot   2022-06-03              150.0               NaN   
11     Onion  A lot   2022-06-04              300.0               NaN   
12     Onion  A lot   2022-06-05              300.0

In [32]:
daily_df["Rolling_Mean_14D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .transform(
        lambda x: x.shift(1).rolling(window=14, min_periods=14).mean()
    )
)

daily_df["Rolling_Std_14D"] = (
    daily_df
    .groupby(["Commodity", "Market"])["Daily_Modal_Price"]
    .transform(
        lambda x: x.shift(1).rolling(window=14, min_periods=14).std()
    )
)

print(
    daily_df[
        [
            "Commodity",
            "Market",
            "Arrival_Date",
            "Daily_Modal_Price",
            "Rolling_Mean_14D",
            "Rolling_Std_14D"
        ]
    ].head(20)
)

print("\nMissing values:")
print(
    daily_df[
        ["Rolling_Mean_14D", "Rolling_Std_14D"]
    ].isna().sum()
)

   Commodity Market Arrival_Date  Daily_Modal_Price  Rolling_Mean_14D  \
0      Onion  A lot   2022-05-24              280.0               NaN   
1      Onion  A lot   2022-05-25              280.0               NaN   
2      Onion  A lot   2022-05-26              200.0               NaN   
3      Onion  A lot   2022-05-27              245.0               NaN   
4      Onion  A lot   2022-05-28              245.0               NaN   
5      Onion  A lot   2022-05-29              220.0               NaN   
6      Onion  A lot   2022-05-30              270.0               NaN   
7      Onion  A lot   2022-05-31              200.0               NaN   
8      Onion  A lot   2022-06-01              245.0               NaN   
9      Onion  A lot   2022-06-02              401.0               NaN   
10     Onion  A lot   2022-06-03              150.0               NaN   
11     Onion  A lot   2022-06-04              300.0               NaN   
12     Onion  A lot   2022-06-05              300.0

In [33]:
daily_df["Month"] = daily_df["Arrival_Date"].dt.month

daily_df["Day_of_Week"] = daily_df["Arrival_Date"].dt.dayofweek

print(
    daily_df[
        [
            "Arrival_Date",
            "Month",
            "Day_of_Week"
        ]
    ].head(15)
)

print("\nMonth values:")
print(sorted(daily_df["Month"].unique()))

print("\nDay_of_Week values:")
print(sorted(daily_df["Day_of_Week"].unique()))

   Arrival_Date  Month  Day_of_Week
0    2022-05-24      5            1
1    2022-05-25      5            2
2    2022-05-26      5            3
3    2022-05-27      5            4
4    2022-05-28      5            5
5    2022-05-29      5            6
6    2022-05-30      5            0
7    2022-05-31      5            1
8    2022-06-01      6            2
9    2022-06-02      6            3
10   2022-06-03      6            4
11   2022-06-04      6            5
12   2022-06-05      6            6
13   2022-06-06      6            0
14   2022-06-07      6            1

Month values:
[np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]

Day_of_Week values:
[np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]


In [34]:
print("Unique commodities:")
print(daily_df["Commodity"].nunique())
print(daily_df["Commodity"].unique())

print("\nUnique markets:")
print(daily_df["Market"].nunique())

Unique commodities:
3
<ArrowStringArray>
['Onion', 'Potato', 'Tomato']
Length: 3, dtype: str

Unique markets:
3031


In [35]:
print("All columns:")
for i, col in enumerate(daily_df.columns, start=1):
    print(f"{i}. {col}")

All columns:
1. Commodity
2. Market
3. Arrival_Date
4. Daily_Modal_Price
5. Price_Return
6. Rolling_Volatility_7D
7. Return_Capped
8. Risk_Level
9. Future_Volatility_7D
10. Future_Risk_Level
11. Lag_1_Price
12. Lag_3_Price
13. Lag_7_Price
14. Price_Change_1D
15. Price_Change_3D
16. Price_Change_7D
17. Rolling_Mean_7D
18. Rolling_Std_7D
19. Rolling_Mean_14D
20. Rolling_Std_14D
21. Month
22. Day_of_Week


In [36]:
feature_columns = [
    "Daily_Modal_Price",
    "Lag_1_Price",
    "Lag_3_Price",
    "Lag_7_Price",
    "Price_Change_1D",
    "Price_Change_3D",
    "Price_Change_7D",
    "Rolling_Mean_7D",
    "Rolling_Std_7D",
    "Rolling_Mean_14D",
    "Rolling_Std_14D",
    "Month",
    "Day_of_Week",
    "Commodity"
]

target_column = "Future_Risk_Level"

print("Number of candidate features:", len(feature_columns))
print("\nCandidate features:")
print(feature_columns)

print("\nTarget:")
print(target_column)

Number of candidate features: 14

Candidate features:
['Daily_Modal_Price', 'Lag_1_Price', 'Lag_3_Price', 'Lag_7_Price', 'Price_Change_1D', 'Price_Change_3D', 'Price_Change_7D', 'Rolling_Mean_7D', 'Rolling_Std_7D', 'Rolling_Mean_14D', 'Rolling_Std_14D', 'Month', 'Day_of_Week', 'Commodity']

Target:
Future_Risk_Level


In [37]:
print("Overall date range:")
print(daily_df["Arrival_Date"].min(), "to", daily_df["Arrival_Date"].max())

print("\nYear-wise observations:")
print(
    daily_df["Arrival_Date"]
    .dt.year
    .value_counts()
    .sort_index()
)

Overall date range:
2022-01-01 00:00:00 to 2026-04-20 00:00:00

Year-wise observations:
Arrival_Date
2022    574441
2023    560790
2024    644972
2025    597523
2026     55874
Name: count, dtype: int64


In [38]:
print(
    daily_df.groupby(daily_df["Arrival_Date"].dt.year)["Future_Risk_Level"]
    .agg(
        Total="size",
        Valid="count",
        Missing=lambda x: x.isna().sum()
    )
)

               Total   Valid  Missing
Arrival_Date                         
2022          574441  570803     3638
2023          560790  558054     2736
2024          644972  641134     3838
2025          597523  576783    20740
2026           55874   37764    18110


In [39]:
valid_target_df = daily_df.dropna(subset=["Future_Risk_Level"])

print("Valid target date range:")
print(
    valid_target_df["Arrival_Date"].min(),
    "to",
    valid_target_df["Arrival_Date"].max()
)

print("\nValid target date range by year:")

print(
    valid_target_df
    .groupby(valid_target_df["Arrival_Date"].dt.year)["Arrival_Date"]
    .agg(["min", "max"])
)

Valid target date range:
2022-01-02 00:00:00 to 2026-04-13 00:00:00

Valid target date range by year:
                    min        max
Arrival_Date                      
2022         2022-01-02 2022-12-31
2023         2023-01-01 2023-12-31
2024         2024-01-01 2024-12-31
2025         2025-01-01 2025-12-31
2026         2026-01-01 2026-04-13


In [40]:
train_df = daily_df[
    (daily_df["Arrival_Date"] < "2025-01-01") &
    (daily_df["Future_Risk_Level"].notna())
].copy()

val_df = daily_df[
    (daily_df["Arrival_Date"] >= "2025-01-01") &
    (daily_df["Arrival_Date"] < "2026-01-01") &
    (daily_df["Future_Risk_Level"].notna())
].copy()

test_df = daily_df[
    (daily_df["Arrival_Date"] >= "2026-01-01") &
    (daily_df["Arrival_Date"] <= "2026-04-13") &
    (daily_df["Future_Risk_Level"].notna())
].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain date range:")
print(train_df["Arrival_Date"].min(), "to", train_df["Arrival_Date"].max())

print("\nValidation date range:")
print(val_df["Arrival_Date"].min(), "to", val_df["Arrival_Date"].max())

print("\nTest date range:")
print(test_df["Arrival_Date"].min(), "to", test_df["Arrival_Date"].max())

Train shape: (1769991, 22)
Validation shape: (576783, 22)
Test shape: (37764, 22)

Train date range:
2022-01-02 00:00:00 to 2024-12-31 00:00:00

Validation date range:
2025-01-01 00:00:00 to 2025-12-31 00:00:00

Test date range:
2026-01-01 00:00:00 to 2026-04-13 00:00:00


In [41]:
print("Train target distribution:")
print(
    train_df["Future_Risk_Level"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nValidation target distribution:")
print(
    val_df["Future_Risk_Level"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTest target distribution:")
print(
    test_df["Future_Risk_Level"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Train target distribution:
Future_Risk_Level
Low       74.81
Medium    15.04
High      10.15
Name: proportion, dtype: float64

Validation target distribution:
Future_Risk_Level
Low       76.22
Medium    14.43
High       9.35
Name: proportion, dtype: float64

Test target distribution:
Future_Risk_Level
Low       65.50
Medium    21.68
High      12.82
Name: proportion, dtype: float64


## Time-Aware Train, Validation, and Test Split

To respect the time-dependent nature of the prediction problem, the dataset was split chronologically rather than randomly.

- Training period: 2022-01-02 to 2024-12-31
- Validation period: 2025-01-01 to 2025-12-31
- Final test period: 2026-01-01 to 2026-04-13

Only observations with an available future 7-observation volatility target were included in the split datasets.

The 2026 period is a partial year and is treated as a final unseen holdout period.

In [42]:
print("Missing values in candidate features — TRAIN:")
print(train_df[feature_columns].isna().sum())

print("\nMissing values in candidate features — VALIDATION:")
print(val_df[feature_columns].isna().sum())

print("\nMissing values in candidate features — TEST:")
print(test_df[feature_columns].isna().sum())

Missing values in candidate features — TRAIN:
Daily_Modal_Price        0
Lag_1_Price              0
Lag_3_Price           8171
Lag_7_Price          24344
Price_Change_1D          0
Price_Change_3D       8171
Price_Change_7D      24344
Rolling_Mean_7D      24344
Rolling_Std_7D       24344
Rolling_Mean_14D     52160
Rolling_Std_14D      52160
Month                    0
Day_of_Week              0
Commodity                0
dtype: int64

Missing values in candidate features — VALIDATION:
Daily_Modal_Price        0
Lag_1_Price              0
Lag_3_Price           4156
Lag_7_Price          10396
Price_Change_1D          0
Price_Change_3D       4156
Price_Change_7D      10396
Rolling_Mean_7D      10396
Rolling_Std_7D       10396
Rolling_Mean_14D     14012
Rolling_Std_14D      14012
Month                    0
Day_of_Week              0
Commodity                0
dtype: int64

Missing values in candidate features — TEST:
Daily_Modal_Price        0
Lag_1_Price              0
Lag_3_Price         

In [43]:
train_model_df = train_df.dropna(
    subset=feature_columns + [target_column]
).copy()

val_model_df = val_df.dropna(
    subset=feature_columns + [target_column]
).copy()

test_model_df = test_df.dropna(
    subset=feature_columns + [target_column]
).copy()

print("Usable modeling rows:")
print("Train:", train_model_df.shape)
print("Validation:", val_model_df.shape)
print("Test:", test_model_df.shape)

print("\nRows removed:")
print("Train:", len(train_df) - len(train_model_df))
print("Validation:", len(val_df) - len(val_model_df))
print("Test:", len(test_df) - len(test_model_df))

Usable modeling rows:
Train: (1717831, 22)
Validation: (562771, 22)
Test: (23913, 22)

Rows removed:
Train: 52160
Validation: 14012
Test: 13851


In [44]:
print("Final feature columns:")
print(feature_columns)

print("\nTarget column:")
print(target_column)

future_related = [
    col for col in feature_columns
    if "Future" in col or "Risk" in col
]

print("\nFuture/Risk columns inside features:")
print(future_related)

Final feature columns:
['Daily_Modal_Price', 'Lag_1_Price', 'Lag_3_Price', 'Lag_7_Price', 'Price_Change_1D', 'Price_Change_3D', 'Price_Change_7D', 'Rolling_Mean_7D', 'Rolling_Std_7D', 'Rolling_Mean_14D', 'Rolling_Std_14D', 'Month', 'Day_of_Week', 'Commodity']

Target column:
Future_Risk_Level

Future/Risk columns inside features:
[]
